In [ ]:
import re
from openai import OpenAI

# Initialize OpenAI client
client = OpenAI()

# -------------------------------
# 1. INPUT VALIDATION GUARDRAIL
# -------------------------------
def validate_input(user_input: str) -> bool:
    if not user_input or len(user_input) > 2000:
        return False

    blocked_patterns = [
        r"ignore previous instructions",
        r"system prompt",
        r"execute code",
        r"<script>",
        r"bypass",
        r"jailbreak"
    ]

    for pattern in blocked_patterns:
        if re.search(pattern, user_input, re.IGNORECASE):
            return False

    return True


# -------------------------------
# 2. MODERATION GUARDRAIL
# -------------------------------
def moderate_input(text: str) -> bool:
    try:
        response = client.moderations.create(
            model="omni-moderation-latest",
            input=text
        )
        return not response.results[0].flagged
    except Exception:
        return False


# -------------------------------
# 3. SIMPLE RAG CONTEXT CHECK
# -------------------------------
def enforce_context(answer: str, context: str) -> bool:
    context_words = set(context.lower().split())
    answer_words = set(answer.lower().split())

    overlap = context_words.intersection(answer_words)

    # Require at least some overlap
    return len(overlap) > 3


# -------------------------------
# 4. OUTPUT FILTER GUARDRAIL
# -------------------------------
def filter_output(response: str) -> str:
    blocked_words = ["hack", "illegal", "expl exploit", "bypass"]

    for word in blocked_words:
        if word in response.lower():
            return "⚠️ Response blocked due to policy restrictions."

    return response


# -------------------------------
# 5. LLM CALL
# -------------------------------
def generate_response(user_input: str, context: str) -> str:
    prompt = f"""
    You are an HR assistant.
    Answer ONLY from the given context.

    Context:
    {context}

    Question:
    {user_input}
    """

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    return response.choices[0].message.content


# -------------------------------
# 6. FULL GUARDRAIL PIPELINE
# -------------------------------
def guardrail_pipeline(user_input: str, context: str) -> str:

    # Step 1: Input validation
    if not validate_input(user_input):
        return "❌ Invalid or unsafe input."

    # Step 2: Moderation
    if not moderate_input(user_input):
        return "❌ Input violates safety policies."

    # Step 3: Generate response
    llm_response = generate_response(user_input, context)

    # Step 4: Context grounding check
    if not enforce_context(llm_response, context):
        return "❌ Response not grounded in knowledge base."

    # Step 5: Output filtering
    safe_response = filter_output(llm_response)

    return safe_response


# -------------------------------
# 7. TEST RUN
# -------------------------------
if __name__ == "__main__":
    user_input = "What is the leave policy?"

    context = """
    Employees are entitled to 20 paid leaves annually.
    Sick leave and casual leave are included in this.
    """

    result = guardrail_pipeline(user_input, context)

    print("\nFinal Response:\n", result)